# RNAQwen v3 GitHub-only Colab

No Google Drive required. The notebook downloads:

- code and the two FSQ checkpoints from this GitHub repository;
- `rnaqa_v3_l234.jsonl` from the GitHub Release asset configured below;
- ERNIE-RNA and Qwen3.6-35B-A3B from their public model repositories.

ERNIE hidden states are streamed into a compact uint16 code cache. Checkpoints
can optionally be synced to a GitHub Release using a Colab secret named
`GITHUB_TOKEN`.

In [ ]:
GITHUB_REPO = 'RomanCohort/notebooktmp'
GITHUB_BRANCH = 'main'
DATASET_RELEASE_TAG = 'dataset-v3'
CHECKPOINT_RELEASE_TAG = 'train-v3-checkpoints'
FSQ_RELEASE_TAG = 'fsq-stream-v1'
DATASET_NAME = 'rnaqa_v3_l234.jsonl'
DATASET_URL = (
    f'https://github.com/{GITHUB_REPO}/releases/download/'
    f'{DATASET_RELEASE_TAG}/{DATASET_NAME}'
)
print('Code:', f'https://github.com/{GITHUB_REPO}')
print('Dataset:', DATASET_URL)

In [ ]:
import subprocess
subprocess.run([
    'nvidia-smi', '--query-gpu=name,memory.total,driver_version',
    '--format=csv,noheader'
], check=False)

In [ ]:
%pip install -q transformers==5.9.0 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2 safetensors==0.8.0 sentencepiece==0.2.2 huggingface_hub==1.33.0 multimolecule==0.2.1
%pip cache purge

In [ ]:
from pathlib import Path
import io, os, shutil, subprocess, sys, zipfile
import requests

ROOT = Path('/content/RNAQwen')
if ROOT.exists():
    shutil.rmtree(ROOT)
archive_url = f'https://codeload.github.com/{GITHUB_REPO}/zip/refs/heads/{GITHUB_BRANCH}'
r = requests.get(archive_url, timeout=(30, 300))
print('GitHub archive:', r.status_code, len(r.content), 'bytes')
if not r.ok:
    raise RuntimeError(f'GitHub archive download failed: {r.status_code} {r.text[:500]}')
tmp_root = Path('/content/_rnaqwen_repo')
if tmp_root.exists():
    shutil.rmtree(tmp_root)
tmp_root.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
    zf.extractall(tmp_root)
inner = next(p for p in tmp_root.iterdir() if p.is_dir())
inner.rename(ROOT)
shutil.rmtree(tmp_root, ignore_errors=True)
ckpt_dir = ROOT / 'checkpoints' / 'fsq_stream'
ckpt_dir.mkdir(parents=True, exist_ok=True)
for name, expected in [('tokenizer.pt', 1_106_665),
                       ('code_emb_contentinit.pt', 46_139_405)]:
    dst = ckpt_dir / name
    if dst.exists() and dst.stat().st_size == expected:
        continue
    url = (f'https://github.com/{GITHUB_REPO}/releases/download/'
           f'{FSQ_RELEASE_TAG}/{name}')
    rr = requests.get(url, timeout=(30, 300))
    print('FSQ asset:', name, rr.status_code, len(rr.content), 'bytes')
    if not rr.ok or len(rr.content) != expected:
        raise RuntimeError(f'FSQ download failed: {name} {rr.status_code} {len(rr.content)}')
    dst.write_bytes(rr.content)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Code ready:', ROOT)

In [ ]:
import requests, time
DATA_DIR = ROOT / 'data' / 'processed'
DATA_DIR.mkdir(parents=True, exist_ok=True)
dataset_path = DATA_DIR / DATASET_NAME
expected_size = 663_240_811

def download_resumable(url, dst, expected):
    part = dst.with_suffix(dst.suffix + '.part')
    if dst.exists() and dst.stat().st_size == expected:
        print('Dataset already downloaded:', dst)
        return
    have = part.stat().st_size if part.exists() else 0
    headers = {'Range': f'bytes={have}-'} if have else {}
    mode = 'ab' if have else 'wb'
    with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as r:
        if r.status_code not in (200, 206):
            raise RuntimeError(f'Dataset download failed: HTTP {r.status_code} {url}')
        if have and r.status_code == 200:
            have, mode = 0, 'wb'
        total = int(r.headers.get('Content-Length', 0)) + have
        with part.open(mode) as fh:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    fh.write(chunk)
                    have += len(chunk)
                    if have % (128 * 1024 * 1024) < len(chunk):
                        print(f'  {have / 2**20:.0f} / {total / 2**20:.0f} MiB', flush=True)
    if part.stat().st_size != expected:
        raise RuntimeError(f'Incomplete dataset: {part.stat().st_size} != {expected}')
    os.replace(part, dst)

download_resumable(DATASET_URL, dataset_path, expected_size)

In [ ]:
from huggingface_hub import snapshot_download
ERNIE_DIR = ROOT / 'models' / 'ernierna'
ERNIE_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id='multimolecule/ernierna',
    local_dir=str(ERNIE_DIR),
)
print('ERNIE-RNA ready:', ERNIE_DIR)

In [ ]:
codes_path = DATA_DIR / '_rnaqa_v3_codes.u16'
codes_index = DATA_DIR / '_rnaqa_v3_codes_index.pt'
free_gb = shutil.disk_usage('/content').free / 2**30
print(f'Free disk: {free_gb:.1f} GiB')
if not codes_index.exists():
    if free_gb < 68.0:
        raise RuntimeError('Need at least 68 GiB free for the quantized 35B model and working space.')
    subprocess.run([
        sys.executable, '-X', 'utf8', '-u',
        str(ROOT / 'scripts/81_encode_v3_codes.py'),
        '--data', str(dataset_path),
        '--encoder', str(ERNIE_DIR),
        '--fsq', str(ROOT / 'checkpoints/fsq_stream/tokenizer.pt'),
        '--out', str(codes_path),
        '--idx', str(codes_index),
        '--pool', '4',
        '--batch', '16',
    ], cwd=ROOT, check=True)
else:
    print('Code cache ready:', codes_path)

In [ ]:
MODEL_DIR = Path('/content/models/Qwen3.6-35B-A3B')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
cfg = MODEL_DIR / 'config.json'
if not cfg.exists():
    subprocess.run([
        sys.executable, '-u', str(ROOT / 'scripts/00b_download_modelscope.py'),
        '--repo', 'Qwen/Qwen3.6-35B-A3B',
        '--dest', str(MODEL_DIR),
        '--workers', '4'
    ], cwd=ROOT, check=True)
index_path = MODEL_DIR / 'model.safetensors.index.json'
if not index_path.exists():
    raise FileNotFoundError(index_path)
import json
index = json.loads(index_path.read_text(encoding='utf-8'))
missing = [n for n in set(index['weight_map'].values()) if not (MODEL_DIR / n).exists()]
if missing:
    raise RuntimeError(f'Incomplete model, missing {len(missing)} shards: {missing[:3]}')
os.environ['RNAQ_QWEN36_DIR'] = str(MODEL_DIR)
os.environ['RNAQ_THREADS'] = '8'
print('Model ready:', MODEL_DIR)

In [ ]:
common = [
    sys.executable, '-X', 'utf8', '-u', str(ROOT / 'scripts/50_routeC_e2e.py'),
    '--data', str(dataset_path),
    '--codes', str(codes_path),
    '--codes-idx', str(codes_index),
    '--fsq', str(ROOT / 'checkpoints/fsq_stream/tokenizer.pt'),
    '--init-code-emb', str(ROOT / 'checkpoints/fsq_stream/code_emb_contentinit.pt'),
    '--base', 'qwen36-35b', '--qwen36-mode', 'int4',
    '--adapter', 'sos', '--lora-r', '8',
    '--token-budget', '512',
]
smoke = common + [
    '--steps', '10', '--min-steps', '10',
    '--eval-every', '10', '--log-every', '1',
    '--eval-n', '2', '--diag', '0', '--ablation-every', '-1',
    '--save-every', '10', '--save-minutes', '5',
    '--tag', 'v3_colab_smoke'
]
rc = subprocess.run(smoke, cwd=ROOT).returncode
print('Smoke exit code:', rc)
if rc != 0:
    raise RuntimeError('Smoke failed; do not start full training.')

In [ ]:
from google.colab import userdata
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
print('Checkpoint sync enabled:', bool(GITHUB_TOKEN))

API = 'https://api.github.com'
HEADERS = {'Accept': 'application/vnd.github+json'}
if GITHUB_TOKEN:
    HEADERS['Authorization'] = f'Bearer {GITHUB_TOKEN}'

CKPT_DIR = ROOT / 'runs' / 'e2e_v3_full' / 'ckpt'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def release_info(create=False):
    url = f'{API}/repos/{GITHUB_REPO}/releases/tags/{CHECKPOINT_RELEASE_TAG}'
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code == 200:
        return r.json()
    if not create or not GITHUB_TOKEN:
        return None
    r = requests.post(
        f'{API}/repos/{GITHUB_REPO}/releases', headers=HEADERS, timeout=30,
        json={'tag_name': CHECKPOINT_RELEASE_TAG,
              'name': 'RNAQwen v3 checkpoints',
              'body': 'Latest resumable training checkpoint.'},
    )
    if r.status_code not in (200, 201):
        raise RuntimeError(f'Create release failed: {r.status_code} {r.text[:300]}')
    return r.json()

def remote_steps():
    rel = release_info(False)
    if not rel:
        return []
    return sorted(
        [a for a in rel.get('assets', []) if a['name'].startswith('step_')],
        key=lambda a: a['name'],
    )

def restore_latest_checkpoint():
    local = sorted(CKPT_DIR.glob('step_*.pt'))
    if local:
        return local[-1]
    remote = remote_steps()
    if not remote:
        return None
    asset = remote[-1]
    dst = CKPT_DIR / asset['name']
    with requests.get(asset['url'], headers={**HEADERS, 'Accept': 'application/octet-stream'},
                      stream=True, timeout=(30, 300)) as r:
        r.raise_for_status()
        with dst.open('wb') as fh:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    fh.write(chunk)
    print('Restored remote checkpoint:', dst)
    return dst

def upload_checkpoint(path):
    if not GITHUB_TOKEN:
        return
    rel = release_info(True)
    existing = next((a for a in remote_steps() if a['name'] == path.name), None)
    if existing and existing.get('size') == path.stat().st_size:
        print('Checkpoint already uploaded:', path.name)
        return
    if existing:
        requests.delete(existing['url'], headers=HEADERS, timeout=30)
    url = rel['upload_url'].split('{', 1)[0]
    with path.open('rb') as fh:
        r = requests.post(
            url, headers={**HEADERS, 'Content-Type': 'application/octet-stream'},
            params={'name': path.name}, data=fh, timeout=(30, 900),
        )
    if r.status_code not in (200, 201):
        raise RuntimeError(f'Checkpoint upload failed: {r.status_code} {r.text[:300]}')
    print('Uploaded checkpoint:', path.name)
    for old in remote_steps():
        if old['name'] != path.name:
            requests.delete(old['url'], headers=HEADERS, timeout=30)

In [ ]:
import shlex, threading
restored = restore_latest_checkpoint()
full = common + [
    '--steps', '20000', '--min-steps', '4000',
    '--early-stop', '1', '--eval-every', '500',
    '--patience', '6', '--min-delta', '0.005',
    '--diag', '1', '--diag-n', '40', '--ablation-every', '-1',
    '--save-every', '100', '--save-minutes', '10', '--keep-ckpt', '3',
    '--tag', 'v3_full'
]
if restored:
    full += ['--resume', str(restored)]
    print('Resuming from:', restored)
else:
    print('Starting a new full run.')

stop_sync = threading.Event()
def sync_loop():
    seen = set()
    while not stop_sync.is_set():
        local = sorted(CKPT_DIR.glob('step_*.pt'))
        if local:
            p = local[-1]
            key = (p.name, p.stat().st_size, int(p.stat().st_mtime))
            if key not in seen:
                try:
                    upload_checkpoint(p)
                    seen.add(key)
                except Exception as exc:
                    print('Checkpoint sync error:', exc, flush=True)
        stop_sync.wait(60)

thread = threading.Thread(target=sync_loop, daemon=True)
thread.start()

quoted = ' '.join(shlex.quote(x) for x in full)
shell_cmd = 'set -o pipefail; timeout --signal=TERM 27000 ' + quoted
print(shell_cmd)
rc = subprocess.run(['bash', '-lc', shell_cmd], cwd=ROOT).returncode
stop_sync.set()
thread.join(timeout=15)
local = sorted(CKPT_DIR.glob('step_*.pt'))
if local:
    upload_checkpoint(local[-1])
print('Full-run segment exit code:', rc)

## Resume after a disconnect

Run the setup cells again, then run the full-training cell. The newest local
or GitHub Release checkpoint is restored automatically.

Without `GITHUB_TOKEN`, checkpoints only survive inside the current Colab
runtime. Add a fine-grained GitHub token with Contents read/write permission
to Colab Secrets under the name `GITHUB_TOKEN` for cross-session resume.